# MJX 04 — From MuJoCo to MJX

### Lab Description

**MJX** is MuJoCo re-implemented in **JAX**. The physics is the same, but the data structures are JAX arrays, so we can `jax.jit` (compile) and `jax.vmap` (batch) the simulation and run **thousands of environments in parallel** on the GPU. This batched throughput is what makes GPU reinforcement learning practical (Module C).

| MuJoCo (C / NumPy) | MJX (JAX) |
|---|---|
| `mujoco.MjModel` | `mjx.put_model(model)` |
| `mujoco.MjData` | `mjx.make_data(mx)` |
| `mujoco.mj_step(m, d)` | `mjx.step(mx, dx)` (jit / vmap-able) |

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Confirm JAX sees the AMD GPU
- Move a MuJoCo model onto the device with `mjx.put_model`
- Run a single jitted MJX rollout with `jax.lax.scan`
- Batch thousands of rollouts with `jax.vmap` and measure throughput

### Import libraries and check the GPU

`jax.devices()` should list a ROCm device — confirmation that JAX will run on the AMD GPU.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jp
import mujoco
from mujoco import mjx

print("JAX devices:", jax.devices())  # expect a RocmDevice on AMD GPU

### Put a MuJoCo model on the device

We build a simple sliding-sphere scene in MJCF, compile it with MuJoCo, then move it to MJX with `mjx.put_model`. `mx` is now a pytree of JAX arrays that lives on the GPU.

In [ ]:
# A sphere that slides on a plane (a simple contact scene).
MJCF = """
<mujoco model="slider">
  <option timestep="0.004"/>
  <worldbody>
    <geom name="floor" type="plane" size="10 10 0.1"/>
    <body name="ball" pos="0 0 0.1">
      <freejoint/>
      <geom name="ball" type="sphere" size="0.1"/>
    </body>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(MJCF)
mx = mjx.put_model(model)
print("qpos dim:", model.nq, "| qvel dim:", model.nv)

### One jitted MJX rollout

`rollout` gives the ball an initial horizontal velocity and steps the physics `N_STEPS` times. We use `jax.lax.scan` (a compiled loop) and `@jax.jit` so the whole rollout becomes a single GPU kernel. The first call compiles; later calls are fast.

In [ ]:
# One MJX rollout: give the ball an initial horizontal velocity and step.
# qvel layout for a freejoint: [vx, vy, vz, wx, wy, wz].
N_STEPS = 200

@jax.jit
def rollout(vx):
    dx = mjx.make_data(mx)
    dx = dx.replace(qvel=dx.qvel.at[0].set(vx))
    def body(dx, _):
        dx = mjx.step(mx, dx)
        return dx, dx.qpos[0]  # track world x of the ball
    dx, xs = jax.lax.scan(body, dx, None, length=N_STEPS)
    return xs

xs = rollout(2.0)
xs.block_until_ready()
plt.figure(figsize=(7, 3))
plt.plot(np.asarray(xs))
plt.xlabel("step"); plt.ylabel("ball x (m)"); plt.title("Single MJX rollout (vx=2.0)"); plt.grid(True)
plt.show()

### Batch thousands of rollouts with `vmap`

`jax.vmap(rollout)` turns the single-environment function into a batched one with no code changes. We time batches of increasing size and print physics steps/second — watch the throughput climb as the GPU fills up.

In [ ]:
# Batch thousands of rollouts with jax.vmap and measure throughput.
batched = jax.jit(jax.vmap(rollout))

def time_batch(n):
    vxs = jp.linspace(0.5, 5.0, n)
    out = batched(vxs); out.block_until_ready()  # compile
    t = time.time(); out = batched(vxs); out.block_until_ready(); dt = time.time() - t
    return dt

sizes = [1, 256, 1024, 4096]
times = [time_batch(n) for n in sizes]
for n, dt in zip(sizes, times):
    print(f"{n:>5} envs x {N_STEPS} steps: {dt*1000:7.1f} ms  ->  {n*N_STEPS/dt/1e6:6.2f} M steps/s")

plt.figure(figsize=(7, 3))
plt.bar([str(n) for n in sizes], [n*N_STEPS/dt/1e6 for n, dt in zip(sizes, times)])
plt.ylabel("M physics steps / s"); plt.xlabel("parallel envs"); plt.title("MJX throughput on GPU")
plt.show()

## Conclusions

A single rollout barely uses the GPU, but `vmap` batching drives millions of physics steps per second. This parallelism is the engine behind GPU RL. MJX 05 uses it for parallel rollouts and **domain randomization**.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT